In [49]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

FIG_DIR = ROOT / "results" / "figures"
TABLE_DIR = ROOT / "results" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
import warnings, math, os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
warnings.filterwarnings("ignore")
import yfinance as yf
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import logging
logging.getLogger("yfinance").setLevel(logging.CRITICAL)


CFG = {
 
    "start": "2010-01-01",
    "end": "2025-12-31", 
    "index": "^NSEI",
    
    
    "horizon": 5,  
    "lookback": 90,  
    
   
    "top_k": 15,  
    "rebalance_weekday": 4,
    "rebalance_threshold": 0.15,  
    

    "train_end": "2023-12-31",  
    "val_start": "2024-01-01",
    "val_end": "2024-06-30",
    "test_start": "2024-07-01",
    "test_end": "2024-12-31",
    "paper_start": "2025-01-01",  
    "paper_end": "2025-12-31",
    

    "min_days": 850,
    
  
    "cost_bps": 8, 
    "slip_bps": 3, 
    
  
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    

    "lstm_epochs": 15,  
    "lstm_batch": 512,  
    "lstm_lr": 5e-4, 
    "lstm_hidden": 96,  
    "lstm_dropout": 0.2, 
    

    "w_xgb": 0.6,  
    "w_lstm": 0.4,  
    

    "use_volatility_sizing": True,
    "use_momentum_filter": True,
    "min_momentum": 0.02,
    "max_volatility": 0.50,
    "sector_max_weight": 0.35,
    "use_stop_loss": True,
    "stop_loss_pct": -0.15,
}

np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["seed"])

print(f"Device: {CFG['device']}")
print(f"PyTorch version: {torch.__version__}")
print(f"\nENHANCED CONFIGURATION LOADED")
print(f"  • Portfolio size: {CFG['top_k']} stocks")
print(f"  • Volatility-based sizing: {CFG['use_volatility_sizing']}")
print(f"  • Momentum filter: {CFG['use_momentum_filter']}")
print(f"  • Stop-loss: {CFG['use_stop_loss']} at {CFG['stop_loss_pct']*100}%")

Device: cpu
PyTorch version: 2.13.0+cpu

ENHANCED CONFIGURATION LOADED
  • Portfolio size: 15 stocks
  • Volatility-based sizing: True
  • Momentum filter: True
  • Stop-loss: True at -15.0%


In [32]:
import os
os.environ.pop("MPLBACKEND", None)

import warnings, math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

import yfinance as yf
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns

import logging
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

In [33]:
# UPDATED: 2025 NIFTY 50 constituents
SYMBOLS_2025 = [
    "ADANIENT.NS", "ADANIPORTS.NS", "APOLLOHOSP.NS", "ASIANPAINT.NS",
    "AXISBANK.NS", "BAJAJ-AUTO.NS", "BAJFINANCE.NS", "BAJAJFINSV.NS",
    "BHARTIARTL.NS", "BPCL.NS", "BRITANNIA.NS", "CIPLA.NS", "COALINDIA.NS",
    "DIVISLAB.NS", "DRREDDY.NS", "EICHERMOT.NS", "GRASIM.NS", "HCLTECH.NS",
    "HDFCBANK.NS", "HDFCLIFE.NS", "HEROMOTOCO.NS", "HINDALCO.NS",
    "HINDUNILVR.NS", "ICICIBANK.NS", "INDUSINDBK.NS", "INFY.NS", "ITC.NS",
    "JSWSTEEL.NS", "KOTAKBANK.NS", "LT.NS", "LTIM.NS", "M&M.NS", "MARUTI.NS",
    "NESTLEIND.NS", "NTPC.NS", "ONGC.NS", "POWERGRID.NS", "RELIANCE.NS",
    "SBILIFE.NS", "SBIN.NS", "SHRIRAMFIN.NS", "SUNPHARMA.NS", "TATACONSUM.NS",
    "TATAMOTORS.NS", "TATASTEEL.NS", "TCS.NS", "TECHM.NS", "TITAN.NS",
    "TRENT.NS", "ULTRACEMCO.NS", "WIPRO.NS"
]

SECTOR_MAP = {
    "ADANIENT.NS": "Infrastructure", "ADANIPORTS.NS": "Infrastructure",
    "APOLLOHOSP.NS": "Healthcare", "ASIANPAINT.NS": "Consumer",
    "AXISBANK.NS": "Banking", "BAJAJ-AUTO.NS": "Auto",
    "BAJFINANCE.NS": "NBFC", "BAJAJFINSV.NS": "NBFC",
    "BHARTIARTL.NS": "Telecom", "BPCL.NS": "Energy",
    "BRITANNIA.NS": "Consumer", "CIPLA.NS": "Pharma",
    "COALINDIA.NS": "Energy", "DIVISLAB.NS": "Pharma",
    "DRREDDY.NS": "Pharma", "EICHERMOT.NS": "Auto",
    "GRASIM.NS": "Diversified", "HCLTECH.NS": "IT",
    "HDFCBANK.NS": "Banking", "HDFCLIFE.NS": "Insurance",
    "HEROMOTOCO.NS": "Auto", "HINDALCO.NS": "Metals",
    "HINDUNILVR.NS": "Consumer", "ICICIBANK.NS": "Banking",
    "INDUSINDBK.NS": "Banking", "INFY.NS": "IT", "ITC.NS": "Consumer",
    "JSWSTEEL.NS": "Metals", "KOTAKBANK.NS": "Banking",
    "LT.NS": "Infrastructure", "LTIM.NS": "IT",
    "M&M.NS": "Auto", "MARUTI.NS": "Auto",
    "NESTLEIND.NS": "Consumer", "NTPC.NS": "Energy",
    "ONGC.NS": "Energy", "POWERGRID.NS": "Energy",
    "RELIANCE.NS": "Energy", "SBILIFE.NS": "Insurance",
    "SBIN.NS": "Banking", "SHRIRAMFIN.NS": "NBFC",
    "SUNPHARMA.NS": "Pharma", "TATACONSUM.NS": "Consumer",
    "TATAMOTORS.NS": "Auto", "TATASTEEL.NS": "Metals",
    "TCS.NS": "IT", "TECHM.NS": "IT",
    "TITAN.NS": "Consumer", "TRENT.NS": "Consumer",
    "ULTRACEMCO.NS": "Cement", "WIPRO.NS": "IT"
}


ALIASES = {
    "MOTHERSUMI.NS": ["MOTHERSON.NS"],
    "SRTRANSFIN.NS": ["SHRIRAMFIN.NS"],
    "TATAGLOBAL.NS": ["TATACONSUM.NS"],
    "INFRATEL.NS": ["INDUSTOWER.NS"],
    "CADILAHC.NS": ["ZYDUSLIFE.NS"],
}

DROP_ALWAYS = set(["RCOM.NS", "DHFL.NS", "VAKRANGEE.NS"])
SYMBOLS = [s for s in SYMBOLS_2025 if s not in DROP_ALWAYS]

def normalize_yf_columns(df):
    """Remove MultiIndex from yfinance dataframes"""
    if df is None or df.empty:
        return df
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy()
        df.columns = df.columns.get_level_values(0)
    return df

def download_one(ticker):
    """Download data for a single ticker"""
    try:
        df = yf.download(ticker, start=CFG["start"], end=CFG["end"], 
                        progress=False, auto_adjust=False)
        df = normalize_yf_columns(df)
        if df is None or df.empty:
            return None
        
        need = ["Open", "High", "Low", "Close", "Volume"]
        if not all(c in df.columns for c in need):
            return None
        
        df = df[need].dropna()
        return None if df.empty else df
    except Exception as e:
        return None

def download_ohlcv(symbols, min_days):
    """Download OHLCV data for all symbols"""
    data, failed = {}, []
    
    print(f"\nDownloading data for {len(symbols)} symbols...")
    for i, s in enumerate(symbols):
        if (i + 1) % 10 == 0:
            print(f"  Progress: {i+1}/{len(symbols)} ({(i+1)/len(symbols)*100:.1f}%)")
        
        tried = [s] + ALIASES.get(s, [])
        ok, used = None, None
        
        for t in tried:
            df = download_one(t)
            if df is not None and len(df) >= min_days:
                ok, used = df, t
                break
        
        if ok is None:
            failed.append(s)
        else:
            data[s] = ok
            if used != s:
                print(f"  Alias used: {s} -> {used}")
    
    print(f"\nDownload complete: {len(data)} successful, {len(failed)} failed")
    if failed:
        print(f"Failed symbols: {', '.join(failed[:10])}")
    
    return data

def get_sector(symbol):
    """Get sector for a symbol"""
    return SECTOR_MAP.get(symbol, "Unknown")

In [34]:


def zscore(x, win=20):
    """Calculate rolling z-score"""
    return (x - x.rolling(win).mean()) / (x.rolling(win).std() + 1e-12)

def compute_features(df):
    """Compute technical indicators and features"""
    out = pd.DataFrame(index=df.index)
    c, h, l, o = df["Close"], df["High"], df["Low"], df["Open"]
    v = df["Volume"].replace(0, np.nan).ffill()
 
    r1 = c.pct_change()
    out["ret1"] = r1
    out["ret3"] = c.pct_change(3)  
    out["ret5"] = c.pct_change(5)
    out["ret10"] = c.pct_change(10)
    out["ret20"] = c.pct_change(20) 
    out["mom20"] = c / c.shift(20) - 1
    out["mom60"] = c / c.shift(60) - 1  
    
  
    out["vol5"] = r1.rolling(5).std()  
    out["vol20"] = r1.rolling(20).std()
    out["vol60"] = r1.rolling(60).std()
    out["vol_ratio"] = out["vol5"] / (out["vol20"] + 1e-8)  
    

    sma10 = c.rolling(10).mean()
    sma20 = c.rolling(20).mean()
    sma50 = c.rolling(50).mean()
    sma200 = c.rolling(200).mean()
    
    out["sma10"] = sma10 / c - 1
    out["sma20"] = sma20 / c - 1
    out["sma50"] = sma50 / c - 1
    out["sma200"] = sma200 / c - 1
    out["sma_cross"] = (sma10 > sma20).astype(float)  # NEW
    

    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    

    tr = pd.DataFrame({
        'hl': h - l,
        'hc': abs(h - c.shift()),
        'lc': abs(l - c.shift())
    }).max(axis=1)
    
    out["tr"] = tr / c
    out["atr14"] = tr.rolling(14).mean() / c
    out["atr_z"] = zscore(out["atr14"], 20)  
    
  
    def calc_rsi(prices, period):
        delta = prices.diff()
        up = delta.clip(lower=0)
        down = (-delta).clip(lower=0)
        rs = up.rolling(period).mean() / (down.rolling(period).mean() + 1e-12)
        return 100 - 100 / (1 + rs)
    
    out["rsi7"] = calc_rsi(c, 7)  
    out["rsi14"] = calc_rsi(c, 14)
    out["rsi21"] = calc_rsi(c, 21)  
  
    macd = ema12 - ema26
    sig = macd.ewm(span=9, adjust=False).mean()
    out["macd"] = macd / c
    out["macd_sig"] = sig / c
    out["macd_hist"] = (macd - sig) / c
    out["macd_cross"] = (macd > sig).astype(float)  
    
   
    bb_mid = c.rolling(20).mean()
    bb_std = c.rolling(20).std()
    out["bb_upper"] = (bb_mid + 2*bb_std) / c - 1
    out["bb_lower"] = (bb_mid - 2*bb_std) / c - 1
    out["bb_width"] = (bb_std * 4) / bb_mid
    out["bb_position"] = (c - bb_mid) / (2 * bb_std)
    
  
    out["logvol_z20"] = zscore(np.log(v + 1), 20)
    out["vol_chg5"] = v.pct_change(5)
    out["vol_sma_ratio"] = v / v.rolling(20).mean() 
  
    out["high_low_ratio"] = (h - l) / c
    out["close_open"] = (c - o) / o
    out["upper_shadow"] = (h - c.where(c > o, o)) / (h - l + 1e-8)
    out["lower_shadow"] = (c.where(c < o, o) - l) / (h - l + 1e-8)
    
   
    out["roc10"] = (c - c.shift(10)) / c.shift(10)
    out["roc20"] = (c - c.shift(20)) / c.shift(20)
    
    return out.replace([np.inf, -np.inf], np.nan)


FEATURES = [
    "ret1", "ret3", "ret5", "ret10", "ret20", "mom20", "mom60",
    "vol5", "vol20", "vol60", "vol_ratio",
    "sma10", "sma20", "sma50", "sma200", "sma_cross",
    "tr", "atr14", "atr_z",
    "rsi7", "rsi14", "rsi21",
    "macd", "macd_sig", "macd_hist", "macd_cross",
    "bb_upper", "bb_lower", "bb_width", "bb_position",
    "logvol_z20", "vol_chg5", "vol_sma_ratio",
    "high_low_ratio", "close_open", "upper_shadow", "lower_shadow",
    "roc10", "roc20"
]

print(f"Total features: {len(FEATURES)}")

Total features: 39


In [36]:


def build_symbol_arrays(data_dict):
    """Create arrays for each symbol and build panel dataset"""
    sym_arrays = {}
    panel_rows = []
    
    for sym, df in data_dict.items():
        feat = compute_features(df)[FEATURES]
        y = (df["Close"].shift(-CFG["horizon"]) / df["Close"] - 1.0).astype(float)
        
        tmp = feat.copy()
        tmp["y"] = y
        tmp["close"] = df["Close"]
        tmp = tmp.dropna(subset=FEATURES + ["y", "close"])
        
        if len(tmp) < (CFG["lookback"] + 300):
            continue
        
        dates = pd.to_datetime(tmp.index)
        X = tmp[FEATURES].values.astype(np.float32)
        yv = tmp["y"].values.astype(np.float32)
        close = tmp["close"].values.astype(np.float32)
        
        sym_arrays[sym] = {
            "dates": dates,
            "X": X,
            "y": yv,
            "close": close
        }
        
        part = pd.DataFrame(X, columns=FEATURES)
        part["y"] = yv
        part["symbol"] = sym
        part["date"] = dates
        panel_rows.append(part)
    
    panel = pd.concat(panel_rows, ignore_index=True)
    panel = panel.sort_values(["date", "symbol"]).reset_index(drop=True)
    
    return sym_arrays, panel

def make_pairs(sym_arrays):
    """Create (symbol, time_index) pairs for LSTM training"""
    pairs = []
    for sym, d in sym_arrays.items():
        for t in range(CFG["lookback"], len(d["dates"])):
            pairs.append((sym, t))
    return pairs

In [37]:


class SeqSliceDataset(Dataset):
    def __init__(self, sym_arrays, pairs, start_date=None, end_date=None, standardize=None):
        self.sym_arrays = sym_arrays
        sd = pd.to_datetime(start_date) if start_date else None
        ed = pd.to_datetime(end_date) if end_date else None
        
        kept = []
        for sym, t in pairs:
            d = sym_arrays[sym]["dates"][t]
            if sd is not None and d < sd:
                continue
            if ed is not None and d > ed:
                continue
            kept.append((sym, t))
        
        self.pairs = kept
        self.mu, self.sig = (standardize if standardize else (None, None))
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, i):
        sym, t = self.pairs[i]
        arr = self.sym_arrays[sym]
        X = arr["X"][t - CFG["lookback"]:t]
        y = arr["y"][t]
        
        if self.mu is not None:
            X = (X - self.mu) / self.sig
        
        date_int = int(arr["dates"][t].value)
        return (
            torch.from_numpy(X),
            torch.tensor([y], dtype=torch.float32),
            sym,
            torch.tensor(date_int, dtype=torch.int64)
        )

class LSTMRegressor(nn.Module):
    def __init__(self, n_features, hidden=64, dropout=0.3):
        super().__init__()
        self.ln = nn.LayerNorm(n_features)
        self.lstm = nn.LSTM(n_features, hidden, batch_first=True, num_layers=2)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        x = self.ln(x)
        x, _ = self.lstm(x)
        x = self.drop(x)
        return self.fc(x[:, -1, :])

def train_lstm(train_loader, val_loader, n_features):
    """Train LSTM model with early stopping"""
    device = CFG["device"]
    model = LSTMRegressor(
        n_features,
        hidden=CFG["lstm_hidden"],
        dropout=CFG["lstm_dropout"]
    ).to(device)
    
    opt = torch.optim.Adam(model.parameters(), lr=CFG["lstm_lr"])
    loss_fn = nn.MSELoss()
    
    best = float("inf")
    best_state = None
    patience = 3
    bad = 0
    
    for ep in range(1, CFG["lstm_epochs"] + 1):
  
        model.train()
        tl = []
        for xb, yb, _, _ in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl.append(loss.item())
        

        model.eval()
        vl = []
        with torch.no_grad():
            for xb, yb, _, _ in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                vl.append(loss_fn(model(xb), yb).item())
        
        v = float(np.mean(vl)) if vl else float("inf")
        print(f"Epoch {ep:02d} | Train MSE: {np.mean(tl):.6f} | Val MSE: {v:.6f}")
        
  
        if v < best - 1e-6:
            best = v
            best_state = {k: vv.detach().cpu().clone() 
                         for k, vv in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f"Early stopping at epoch {ep}")
                break
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    return model

@torch.no_grad()
def predict_lstm(model, loader):
    """Generate LSTM predictions"""
    device = CFG["device"]
    model.eval()
    out = []
    
    for xb, _, syms, date_int in loader:
        xb = xb.to(device)
        pred = model(xb).detach().cpu().numpy().reshape(-1)
        dates = pd.to_datetime(date_int.numpy(), unit="ns")
        out.extend(list(zip(dates, list(syms), pred)))
    
    df = pd.DataFrame(out, columns=["date", "symbol", "pred_lstm"])
    df["date"] = pd.to_datetime(df["date"])
    return df

In [38]:


def calculate_metrics(y_true, y_pred, prefix=""):
    """Calculate regression metrics"""
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    

    direction_acc = np.mean(np.sign(y_true) == np.sign(y_pred))
    
    metrics = {
        f"{prefix}MSE": mse,
        f"{prefix}RMSE": rmse,
        f"{prefix}MAE": mae,
        f"{prefix}R2": r2,
        f"{prefix}Direction_Acc": direction_acc
    }
    
    return metrics

def sharpe(returns, ann=252):
    """Calculate Sharpe ratio"""
    r = np.asarray(returns, float)
    if len(r) < 2 or r.std() == 0:
        return 0.0
    return (r.mean() / r.std()) * math.sqrt(ann)

def max_drawdown(equity):
    """Calculate maximum drawdown"""
    e = np.asarray(equity, float)
    peak = np.maximum.accumulate(e)
    dd = (e / peak) - 1
    return float(dd.min())

def cagr(equity, ann=252):
    """Calculate Compound Annual Growth Rate"""
    e = np.asarray(equity, float)
    if len(e) < 2:
        return 0.0
    years = (len(e) - 1) / ann
    if years <= 0:
        return 0.0
    return float((e[-1] / e[0]) ** (1 / years) - 1)

def calmar_ratio(equity, ann=252):
    """Calculate Calmar ratio (CAGR / Max Drawdown)"""
    c = cagr(equity, ann)
    mdd = abs(max_drawdown(equity))
    return c / mdd if mdd > 0 else 0.0

def sortino_ratio(returns, ann=252):
    """Calculate Sortino ratio"""
    r = np.asarray(returns, float)
    if len(r) < 2:
        return 0.0
    downside = r[r < 0].std()
    if downside == 0:
        return 0.0
    return (r.mean() / downside) * math.sqrt(ann)

In [39]:


def get_rebalance_dates(dates, weekday=4):
    """Get rebalancing dates"""
    idx = pd.DatetimeIndex(sorted(pd.to_datetime(dates).unique()))
    return idx[idx.weekday == weekday]

def calculate_volatility_weights(today_data, sym_arrays, d):
    """Calculate position sizes based on inverse volatility"""
    if not CFG["use_volatility_sizing"]:
        return None
    
    vol_weights = {}
    for _, row in today_data.iterrows():
        sym = row["symbol"]
        arr = sym_arrays.get(sym)
        if arr is None:
            continue
        
        dt = arr["dates"].values
        t = np.searchsorted(dt, np.datetime64(d))
        if t < 20 or t >= len(arr["close"]):
            continue
 
        returns = np.diff(arr["close"][max(0, t-20):t]) / arr["close"][max(0, t-20):t-1]
        vol = np.std(returns) * np.sqrt(252)
        
        if vol > 0:
            vol_weights[sym] = 1.0 / vol

    if vol_weights:
        total = sum(vol_weights.values())
        vol_weights = {k: v/total for k, v in vol_weights.items()}
    
    return vol_weights

def apply_sector_diversification(today_data, current_weights):
    """Ensure no sector is overweighted"""
    if not CFG.get("sector_max_weight"):
        return current_weights
    
    sector_exposure = {}
    for sym, w in current_weights.items():
        sector = get_sector(sym)
        sector_exposure[sector] = sector_exposure.get(sector, 0) + w

    adjusted_weights = current_weights.copy()
    for sector, exposure in sector_exposure.items():
        if exposure > CFG["sector_max_weight"]:

            sector_stocks = [s for s in current_weights.keys() if get_sector(s) == sector]
            scale = CFG["sector_max_weight"] / exposure
            for s in sector_stocks:
                adjusted_weights[s] *= scale

    total = sum(adjusted_weights.values())
    if total > 0:
        adjusted_weights = {k: v/total for k, v in adjusted_weights.items()}
    
    return adjusted_weights

def run_backtest_enhanced(panel_pred, sym_arrays, idx_close, start_date, end_date, name="Backtest"):
    """
    ADVANCED backtest with profit-maximizing features
    """
    panel_pred = panel_pred.copy()
    panel_pred["date"] = pd.to_datetime(panel_pred["date"])
    
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    
    dates = pd.DatetimeIndex(sorted(panel_pred["date"].unique()))
    dates = dates[(dates >= start) & (dates <= end)]
    
    if len(dates) == 0:
        print(f"Warning: No dates in range {start} to {end}")
        return None
    
    rebal = set(get_rebalance_dates(dates, CFG["rebalance_weekday"]))
    
    equity = [1.0]
    eq_dates = [dates[0]]
    daily = []
    pos = {}
    trades = []
    holdings_history = []
    detailed_transactions = []
    stop_loss_triggers = []
    
    print(f"\nRunning {name} backtest...")
    print(f"   Period: {dates[0].date()} to {dates[-1].date()}")
    print(f"   Trading days: {len(dates)}")
    
    for i in range(1, len(dates)):
        d = dates[i]
        prev = dates[i - 1]
 
        day = 0.0
        stop_loss_hit = []
        
        for sym, w in list(pos.items()):
            arr = sym_arrays.get(sym)
            if arr is None:
                continue
            
            dt = arr["dates"].values
            t0 = np.searchsorted(dt, np.datetime64(prev))
            t1 = np.searchsorted(dt, np.datetime64(d))
            
            if t0 >= len(arr["close"]) or t1 >= len(arr["close"]):
                continue
            if dt[t0] != np.datetime64(prev) or dt[t1] != np.datetime64(d):
                continue
            
            day_return = float(arr["close"][t1] / arr["close"][t0] - 1.0)
            
    
            if CFG["use_stop_loss"] and day_return < CFG["stop_loss_pct"]:
                stop_loss_hit.append((sym, day_return))
                day += w * day_return
                detailed_transactions.append({
                    'date': d,
                    'action': 'STOP_LOSS',
                    'symbol': sym,
                    'weight': w,
                    'return': day_return,
                    'value': equity[-1] * w,
                    'portfolio_value': equity[-1]
                })
                del pos[sym]
            else:
                day += w * day_return
        
        equity.append(equity[-1] * (1 + day))
        eq_dates.append(d)
        daily.append(day)
        
        if stop_loss_hit:
            stop_loss_triggers.extend(stop_loss_hit)
        

        if d in rebal:
            today = panel_pred[panel_pred["date"] == d].copy()

            if CFG["use_momentum_filter"]:
                today = today[today["mom20"].fillna(0) >= CFG["min_momentum"]]
            
            if CFG.get("max_volatility"):
                today = today[today["vol60"].fillna(1.0) <= CFG["max_volatility"]]
            
            today = today.dropna(subset=["score"])
            today = today.sort_values("score", ascending=False)
            
            if len(today) >= CFG["top_k"]:
                picks = today["symbol"].head(CFG["top_k"]).tolist()
                
                if CFG["use_volatility_sizing"]:
                    vol_weights = calculate_volatility_weights(today.head(CFG["top_k"]), sym_arrays, d)
                    if vol_weights:
                        new_pos = vol_weights
                    else:
                        w = 1.0 / len(picks)
                        new_pos = {s: w for s in picks}
                else:
                    w = 1.0 / len(picks)
                    new_pos = {s: w for s in picks}

                new_pos = apply_sector_diversification(today.head(CFG["top_k"]), new_pos)
                
                old_stocks = set(pos.keys())
                new_stocks = set(new_pos.keys())
                
                union = set(pos) | set(new_pos)
                potential_turnover = sum(abs(new_pos.get(s, 0.0) - pos.get(s, 0.0)) for s in union)
     
                if potential_turnover < CFG.get("rebalance_threshold", 0):
                    continue
                
                sells = old_stocks - new_stocks
                buys = new_stocks - old_stocks
                holds = old_stocks & new_stocks
                
                portfolio_value = equity[-1]
          
                for stock in sells:
                    detailed_transactions.append({
                        'date': d,
                        'action': 'SELL',
                        'symbol': stock,
                        'weight': pos[stock],
                        'value': portfolio_value * pos[stock],
                        'portfolio_value': portfolio_value
                    })
                
                for stock in buys:
                    detailed_transactions.append({
                        'date': d,
                        'action': 'BUY',
                        'symbol': stock,
                        'weight': new_pos[stock],
                        'value': portfolio_value * new_pos[stock],
                        'portfolio_value': portfolio_value
                    })
                
           
                cost = (CFG["cost_bps"] + CFG["slip_bps"]) * 1e-4 * potential_turnover
                equity[-1] *= (1 - cost)
                
                trades.append({
                    "date": d,
                    "action": "rebalance",
                    "turnover": potential_turnover,
                    "cost": cost,
                    "holdings": picks,
                    "buys": list(buys),
                    "sells": list(sells),
                    "holds": list(holds),
                    "num_buys": len(buys),
                    "num_sells": len(sells)
                })
                
                pos = new_pos
                holdings_history.append({"date": d, "holdings": picks})
    

    idx_close = idx_close.dropna()
    idx_ret = idx_close.reindex(eq_dates).pct_change().fillna(0.0).values
    idx_eq = [1.0]
    for r in idx_ret[1:]:
        idx_eq.append(idx_eq[-1] * (1 + float(r)))

    ann_vol = np.std(daily) * np.sqrt(252) * 100 if daily else 0
    
    stats = {
        "Period": name,
        "Start": eq_dates[0].strftime("%Y-%m-%d"),
        "End": eq_dates[-1].strftime("%Y-%m-%d"),
        "Total_Return": (equity[-1] - 1.0) * 100,
        "CAGR": cagr(equity) * 100,
        "Sharpe": sharpe(daily),
        "Sortino": sortino_ratio(daily),
        "MaxDD": max_drawdown(equity) * 100,
        "Calmar": calmar_ratio(equity),
        "Volatility": ann_vol,
        "Num_Trades": len(trades),
        "Avg_Turnover": np.mean([t["turnover"] for t in trades]) * 100 if trades else 0,
        "Total_Cost": sum([t["cost"] for t in trades]) * 100,
        "Best_Day": max(daily) * 100 if daily else 0,
        "Worst_Day": min(daily) * 100 if daily else 0,
        "Win_Rate": np.mean([d > 0 for d in daily]) * 100 if daily else 0,
        "Benchmark_Return": (idx_eq[-1] - 1.0) * 100,
        "Benchmark_CAGR": cagr(idx_eq) * 100,
        "Benchmark_Sharpe": sharpe(idx_ret[1:]),
        "Benchmark_MaxDD": max_drawdown(idx_eq) * 100,
        "Alpha": (equity[-1] - idx_eq[-1]) * 100,
        "Final_Portfolio_Value": equity[-1],
        "Total_Buys": sum([t["num_buys"] for t in trades]),
        "Total_Sells": sum([t["num_sells"] for t in trades]),
        "Stop_Loss_Triggers": len(stop_loss_triggers)
    }
    
    print(f"   Return: {stats['Total_Return']:.2f}%")
    print(f"   Sharpe: {stats['Sharpe']:.2f}")
    print(f"   Max DD: {stats['MaxDD']:.2f}%")
    print(f"   Trades: {stats['Num_Trades']}")
    
    return {
        "dates": eq_dates,
        "equity": equity,
        "idx_equity": idx_eq,
        "daily_returns": daily,
        "stats": stats,
        "trades": trades,
        "holdings_history": holdings_history,
        "detailed_transactions": detailed_transactions,
        "stop_loss_triggers": stop_loss_triggers
    }

In [40]:


def get_rebalance_dates(dates, weekday=4):
    """Get rebalancing dates"""
    idx = pd.DatetimeIndex(sorted(pd.to_datetime(dates).unique()))
    return idx[idx.weekday == weekday]

def calculate_volatility_weights(today_data, sym_arrays, d):
    """Calculate position sizes based on inverse volatility"""
    if not CFG["use_volatility_sizing"]:
        return None
    
    vol_weights = {}
    for _, row in today_data.iterrows():
        sym = row["symbol"]
        arr = sym_arrays.get(sym)
        if arr is None:
            continue
        
        dt = arr["dates"].values
        t = np.searchsorted(dt, np.datetime64(d))
        if t < 20 or t >= len(arr["close"]):
            continue
  
        returns = np.diff(arr["close"][max(0, t-20):t]) / arr["close"][max(0, t-20):t-1]
        vol = np.std(returns) * np.sqrt(252)
        
        if vol > 0:
            vol_weights[sym] = 1.0 / vol
    

    if vol_weights:
        total = sum(vol_weights.values())
        vol_weights = {k: v/total for k, v in vol_weights.items()}
    
    return vol_weights

def apply_sector_diversification(today_data, current_weights):
    """Ensure no sector is overweighted"""
    if not CFG.get("sector_max_weight"):
        return current_weights
    
    sector_exposure = {}
    for sym, w in current_weights.items():
        sector = get_sector(sym)
        sector_exposure[sector] = sector_exposure.get(sector, 0) + w
    

    adjusted_weights = current_weights.copy()
    for sector, exposure in sector_exposure.items():
        if exposure > CFG["sector_max_weight"]:
            sector_stocks = [s for s in current_weights.keys() if get_sector(s) == sector]
            scale = CFG["sector_max_weight"] / exposure
            for s in sector_stocks:
                adjusted_weights[s] *= scale

    total = sum(adjusted_weights.values())
    if total > 0:
        adjusted_weights = {k: v/total for k, v in adjusted_weights.items()}
    
    return adjusted_weights

def run_backtest_enhanced(panel_pred, sym_arrays, idx_close, start_date, end_date, name="Backtest"):
    """
    Enhanced backtest with profit-maximizing features
    """
    panel_pred = panel_pred.copy()
    panel_pred["date"] = pd.to_datetime(panel_pred["date"])
    
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    
    dates = pd.DatetimeIndex(sorted(panel_pred["date"].unique()))
    dates = dates[(dates >= start) & (dates <= end)]
    
    if len(dates) == 0:
        print(f"Warning: No dates in range {start} to {end}")
        return None
    
    rebal = set(get_rebalance_dates(dates, CFG["rebalance_weekday"]))
    
    equity = [1.0]
    eq_dates = [dates[0]]
    daily = []
    pos = {}
    trades = []
    holdings_history = []
    detailed_transactions = []
    stop_loss_triggers = []
    
    print(f"\n Running {name} backtest...")
    print(f"   Period: {dates[0].date()} to {dates[-1].date()}")
    print(f"   Trading days: {len(dates)}")
    
    for i in range(1, len(dates)):
        d = dates[i]
        prev = dates[i - 1]
    
        day = 0.0
        stop_loss_hit = []
        
        for sym, w in list(pos.items()):
            arr = sym_arrays.get(sym)
            if arr is None:
                continue
            
            dt = arr["dates"].values
            t0 = np.searchsorted(dt, np.datetime64(prev))
            t1 = np.searchsorted(dt, np.datetime64(d))
            
            if t0 >= len(arr["close"]) or t1 >= len(arr["close"]):
                continue
            if dt[t0] != np.datetime64(prev) or dt[t1] != np.datetime64(d):
                continue
            
            day_return = float(arr["close"][t1] / arr["close"][t0] - 1.0)
            
  
            if CFG["use_stop_loss"] and day_return < CFG["stop_loss_pct"]:
                stop_loss_hit.append((sym, day_return))
                day += w * day_return
                detailed_transactions.append({
                    'date': d,
                    'action': 'STOP_LOSS',
                    'symbol': sym,
                    'weight': w,
                    'return': day_return,
                    'value': equity[-1] * w,
                    'portfolio_value': equity[-1]
                })
                del pos[sym]
            else:
                day += w * day_return
        
        equity.append(equity[-1] * (1 + day))
        eq_dates.append(d)
        daily.append(day)
        
        if stop_loss_hit:
            stop_loss_triggers.extend(stop_loss_hit)

        if d in rebal:
            today = panel_pred[panel_pred["date"] == d].copy()
            
            if CFG["use_momentum_filter"]:
                today = today[today["mom20"].fillna(0) >= CFG["min_momentum"]]

            if CFG.get("max_volatility"):
                today = today[today["vol60"].fillna(1.0) <= CFG["max_volatility"]]
            
            today = today.dropna(subset=["score"])
            today = today.sort_values("score", ascending=False)
            
            if len(today) >= CFG["top_k"]:
                picks = today["symbol"].head(CFG["top_k"]).tolist()
     
                if CFG["use_volatility_sizing"]:
                    vol_weights = calculate_volatility_weights(today.head(CFG["top_k"]), sym_arrays, d)
                    if vol_weights:
                        new_pos = vol_weights
                    else:
                        w = 1.0 / len(picks)
                        new_pos = {s: w for s in picks}
                else:
                    w = 1.0 / len(picks)
                    new_pos = {s: w for s in picks}

                new_pos = apply_sector_diversification(today.head(CFG["top_k"]), new_pos)
   
                old_stocks = set(pos.keys())
                new_stocks = set(new_pos.keys())
                
                union = set(pos) | set(new_pos)
                potential_turnover = sum(abs(new_pos.get(s, 0.0) - pos.get(s, 0.0)) for s in union)
                
                if potential_turnover < CFG.get("rebalance_threshold", 0):
                    continue
                
                sells = old_stocks - new_stocks
                buys = new_stocks - old_stocks
                holds = old_stocks & new_stocks
                
                portfolio_value = equity[-1]

                for stock in sells:
                    detailed_transactions.append({
                        'date': d,
                        'action': 'SELL',
                        'symbol': stock,
                        'weight': pos[stock],
                        'value': portfolio_value * pos[stock],
                        'portfolio_value': portfolio_value
                    })
                
                for stock in buys:
                    detailed_transactions.append({
                        'date': d,
                        'action': 'BUY',
                        'symbol': stock,
                        'weight': new_pos[stock],
                        'value': portfolio_value * new_pos[stock],
                        'portfolio_value': portfolio_value
                    })
                
                cost = (CFG["cost_bps"] + CFG["slip_bps"]) * 1e-4 * potential_turnover
                equity[-1] *= (1 - cost)
                
                trades.append({
                    "date": d,
                    "action": "rebalance",
                    "turnover": potential_turnover,
                    "cost": cost,
                    "holdings": picks,
                    "buys": list(buys),
                    "sells": list(sells),
                    "holds": list(holds),
                    "num_buys": len(buys),
                    "num_sells": len(sells)
                })
                
                pos = new_pos
                holdings_history.append({"date": d, "holdings": picks})

    idx_close = idx_close.dropna()
    idx_ret = idx_close.reindex(eq_dates).pct_change().fillna(0.0).values
    idx_eq = [1.0]
    for r in idx_ret[1:]:
        idx_eq.append(idx_eq[-1] * (1 + float(r)))

    ann_vol = np.std(daily) * np.sqrt(252) * 100 if daily else 0
    
    stats = {
        "Period": name,
        "Start": eq_dates[0].strftime("%Y-%m-%d"),
        "End": eq_dates[-1].strftime("%Y-%m-%d"),
        "Total_Return": (equity[-1] - 1.0) * 100,
        "CAGR": cagr(equity) * 100,
        "Sharpe": sharpe(daily),
        "Sortino": sortino_ratio(daily),
        "MaxDD": max_drawdown(equity) * 100,
        "Calmar": calmar_ratio(equity),
        "Volatility": ann_vol,
        "Num_Trades": len(trades),
        "Avg_Turnover": np.mean([t["turnover"] for t in trades]) * 100 if trades else 0,
        "Total_Cost": sum([t["cost"] for t in trades]) * 100,
        "Best_Day": max(daily) * 100 if daily else 0,
        "Worst_Day": min(daily) * 100 if daily else 0,
        "Win_Rate": np.mean([d > 0 for d in daily]) * 100 if daily else 0,
        "Benchmark_Return": (idx_eq[-1] - 1.0) * 100,
        "Benchmark_CAGR": cagr(idx_eq) * 100,
        "Benchmark_Sharpe": sharpe(idx_ret[1:]),
        "Benchmark_MaxDD": max_drawdown(idx_eq) * 100,
        "Alpha": (equity[-1] - idx_eq[-1]) * 100,
        "Final_Portfolio_Value": equity[-1],
        "Total_Buys": sum([t["num_buys"] for t in trades]),
        "Total_Sells": sum([t["num_sells"] for t in trades]),
        "Stop_Loss_Triggers": len(stop_loss_triggers)
    }
    
    print(f"   ✓ Return: {stats['Total_Return']:.2f}%")
    print(f"   ✓ Sharpe: {stats['Sharpe']:.2f}")
    print(f"   ✓ Max DD: {stats['MaxDD']:.2f}%")
    print(f"   ✓ Trades: {stats['Num_Trades']}")
    
    return {
        "dates": eq_dates,
        "equity": equity,
        "idx_equity": idx_eq,
        "daily_returns": daily,
        "stats": stats,
        "trades": trades,
        "holdings_history": holdings_history,
        "detailed_transactions": detailed_transactions,
        "stop_loss_triggers": stop_loss_triggers
    }


def plot_equity_curves(results_dict, title="Portfolio Performance"):
    """Plot equity curves for multiple periods"""
    valid_results = {k: v for k, v in results_dict.items() if v is not None}
    if not valid_results:
        print("No valid results to plot")
        return
    
    n_plots = len(valid_results)
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    for idx, (name, res) in enumerate(valid_results.items()):
        if idx >= 4:
            break
        
        ax = axes[idx]
        ax.plot(res["dates"], res["equity"], label="Strategy", linewidth=2, color='#2E86AB')
        ax.plot(res["dates"], res["idx_equity"], label="NIFTY 50", linewidth=2, 
                color='#A23B72', linestyle='--')
        
        ax.set_title(f"{name} ({res['stats']['Start']} to {res['stats']['End']})", 
                    fontsize=12, fontweight='bold')
        ax.set_xlabel("Date")
        ax.set_ylabel("Portfolio Value")
        ax.legend(loc='upper left')
        ax.grid(True, alpha=0.3)
        
   
        text = f"Return: {res['stats']['Total_Return']:.1f}%\n"
        text += f"Sharpe: {res['stats']['Sharpe']:.2f}\n"
        text += f"MaxDD: {res['stats']['MaxDD']:.1f}%"
        ax.text(0.02, 0.98, text, transform=ax.transAxes, 
               verticalalignment='top', bbox=dict(boxstyle='round', 
               facecolor='wheat', alpha=0.5), fontsize=9)

    for idx in range(len(valid_results), 4):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('equity_curves.png', dpi=300, bbox_inches='tight')
    plt.show()

def plot_monthly_returns(results_dict):
    """Plot monthly returns heatmap - FIXED VERSION"""
    valid_results = {k: v for k, v in results_dict.items() if v is not None}
    if not valid_results:
        print("No valid results to plot")
        return
    
    n_plots = len(valid_results)
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    for idx, (name, res) in enumerate(valid_results.items()):
        if idx >= 4:
            break
        

        equity_series = pd.Series(res["equity"], index=pd.DatetimeIndex(res["dates"]))
        monthly = equity_series.resample('ME').last().pct_change().dropna() * 100
        
        if len(monthly) == 0:
            axes[idx].text(0.5, 0.5, f'{name}\nInsufficient data for monthly returns', 
                          ha='center', va='center', transform=axes[idx].transAxes)
            continue

        df_monthly = pd.DataFrame({
            'Year': monthly.index.year,
            'Month': monthly.index.month,
            'Return': monthly.values
        })
        pivot = df_monthly.pivot(index='Month', columns='Year', values='Return')
        

        if not pivot.empty and pivot.shape[0] > 0 and pivot.shape[1] > 0:
            ax = axes[idx]
            sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
                       ax=ax, cbar_kws={'label': 'Return (%)'}, linewidths=0.5)
            ax.set_title(f"{name} - Monthly Returns (%)", fontsize=12, fontweight='bold')
            ax.set_ylabel("Month")
            ax.set_xlabel("Year")
        else:
            axes[idx].text(0.5, 0.5, f'{name}\nInsufficient data', 
                          ha='center', va='center', transform=axes[idx].transAxes)
    

    for idx in range(len(valid_results), 4):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('monthly_returns_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()

def plot_drawdown(results_dict):
    """Plot drawdown curves"""
    valid_results = {k: v for k, v in results_dict.items() if v is not None}
    if not valid_results:
        print("No valid results to plot")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    for idx, (name, res) in enumerate(valid_results.items()):
        if idx >= 4:
            break
        
        equity = np.array(res["equity"])
        peak = np.maximum.accumulate(equity)
        drawdown = (equity / peak - 1) * 100
        
        ax = axes[idx]
        ax.fill_between(range(len(drawdown)), drawdown, 0, alpha=0.3, color='red')
        ax.plot(drawdown, color='darkred', linewidth=1.5)
        ax.set_title(f"{name} - Drawdown", fontsize=12, fontweight='bold')
        ax.set_xlabel("Days")
        ax.set_ylabel("Drawdown (%)")
        ax.grid(True, alpha=0.3)
        ax.axhline(y=res['stats']['MaxDD'], color='black', linestyle='--', 
                  label=f"Max DD: {res['stats']['MaxDD']:.1f}%")
        ax.legend()

    for idx in range(len(valid_results), 4):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('drawdown_curves.png', dpi=300, bbox_inches='tight')
    plt.show()

def plot_returns_distribution(results_dict):
    """Plot daily returns distribution"""
    valid_results = {k: v for k, v in results_dict.items() if v is not None}
    if not valid_results:
        print("No valid results to plot")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    for idx, (name, res) in enumerate(valid_results.items()):
        if idx >= 4:
            break
        
        returns = np.array(res["daily_returns"]) * 100
        
        ax = axes[idx]
        ax.hist(returns, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
        ax.axvline(returns.mean(), color='red', linestyle='--', 
                  linewidth=2, label=f'Mean: {returns.mean():.2f}%')
        ax.axvline(0, color='black', linestyle='-', linewidth=1)
        ax.set_title(f"{name} - Daily Returns Distribution", fontsize=12, fontweight='bold')
        ax.set_xlabel("Daily Return (%)")
        ax.set_ylabel("Frequency")
        ax.legend()
        ax.grid(True, alpha=0.3)
        

        text = f"Std: {returns.std():.2f}%\n"
        text += f"Skew: {pd.Series(returns).skew():.2f}\n"
        text += f"Kurt: {pd.Series(returns).kurtosis():.2f}"
        ax.text(0.02, 0.98, text, transform=ax.transAxes,
               verticalalignment='top', bbox=dict(boxstyle='round',
               facecolor='wheat', alpha=0.5), fontsize=9)

    for idx in range(len(valid_results), 4):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('returns_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()

def plot_rolling_metrics(results_dict, window=60):
    """Plot rolling Sharpe and volatility"""
    valid_results = {k: v for k, v in results_dict.items() if v is not None}
    if not valid_results:
        print("No valid results to plot")
        return
    
    n_results = len(valid_results)
    fig, axes = plt.subplots(n_results, 2, figsize=(16, 4*n_results))
    
    if n_results == 1:
        axes = axes.reshape(1, -1)
    
    for idx, (name, res) in enumerate(valid_results.items()):
        returns = pd.Series(res["daily_returns"], index=pd.DatetimeIndex(res["dates"][1:]))
        

        rolling_sharpe = returns.rolling(window).mean() / returns.rolling(window).std() * np.sqrt(252)
        axes[idx, 0].plot(rolling_sharpe.index, rolling_sharpe.values, linewidth=1.5)
        axes[idx, 0].set_title(f"{name} - Rolling {window}-Day Sharpe Ratio", fontweight='bold')
        axes[idx, 0].axhline(y=0, color='black', linestyle='--', alpha=0.3)
        axes[idx, 0].grid(True, alpha=0.3)
        axes[idx, 0].set_ylabel("Sharpe Ratio")
        

        rolling_vol = returns.rolling(window).std() * np.sqrt(252) * 100
        axes[idx, 1].plot(rolling_vol.index, rolling_vol.values, linewidth=1.5, color='orange')
        axes[idx, 1].set_title(f"{name} - Rolling {window}-Day Volatility", fontweight='bold')
        axes[idx, 1].grid(True, alpha=0.3)
        axes[idx, 1].set_ylabel("Annualized Volatility (%)")
    
    plt.tight_layout()
    plt.savefig('rolling_metrics.png', dpi=300, bbox_inches='tight')
    plt.show()



In [41]:


def analyze_trades(results_dict):
    """Analyze trade statistics"""
    trade_analysis = {}
    
    for name, res in results_dict.items():
        if res is None:
            continue
        
        trades = res['trades']
        if not trades:
            continue
        
        turnovers = [t['turnover'] for t in trades]
        costs = [t['cost'] for t in trades]
        
        analysis = {
            'Total_Trades': len(trades),
            'Avg_Turnover': np.mean(turnovers) * 100,
            'Max_Turnover': np.max(turnovers) * 100,
            'Min_Turnover': np.min(turnovers) * 100,
            'Total_Cost': sum(costs) * 100,
            'Avg_Cost_Per_Trade': np.mean(costs) * 100,
            'Rebalance_Frequency_Days': len(res['daily_returns']) / len(trades) if trades else 0
        }
        
        trade_analysis[name] = analysis

    df = pd.DataFrame(trade_analysis).T
    print("\n" + "="*80)
    print("TRADE ANALYSIS")
    print("="*80)
    print(df.to_string())
    print("="*80)
    
    return df

def analyze_holdings(results_dict):
    """Analyze holding patterns"""
    for name, res in results_dict.items():
        if res is None or not res.get('holdings_history'):
            continue
        
        print(f"\n{'='*80}")
        print(f"{name} - HOLDINGS ANALYSIS")
        print('='*80)

        all_holdings = []
        for h in res['holdings_history']:
            all_holdings.extend(h['holdings'])
        
        holding_counts = pd.Series(all_holdings).value_counts()
        
        print(f"\nTop 15 Most Held Stocks:")
        print("-" * 80)
        for stock, count in holding_counts.head(15).items():
            pct = (count / len(res['holdings_history'])) * 100
            print(f"{stock:20s} : {count:3d} times ({pct:5.1f}% of rebalances)")
        
        print(f"\nTotal Unique Stocks Held: {len(holding_counts)}")
        print(f"Average Holdings per Rebalance: {len(all_holdings)/len(res['holdings_history']):.1f}")

In [42]:


def create_performance_summary(results_dict):
    """Create comprehensive performance summary table"""
    summary = {}
    
    for name, res in results_dict.items():
        if res is None:
            continue
        
        stats = res['stats']
        summary[name] = {
            'Period': f"{stats['Start']} to {stats['End']}",
            'Total Return (%)': stats['Total_Return'],
            'CAGR (%)': stats['CAGR'],
            'Sharpe Ratio': stats['Sharpe'],
            'Sortino Ratio': stats['Sortino'],
            'Max Drawdown (%)': stats['MaxDD'],
            'Calmar Ratio': stats['Calmar'],
            'Volatility (%)': np.std(res['daily_returns']) * np.sqrt(252) * 100,
            'Win Rate (%)': stats['Win_Rate'],
            'Best Day (%)': stats['Best_Day'],
            'Worst Day (%)': stats['Worst_Day'],
            'Num Trades': stats['Num_Trades'],
            'Benchmark Return (%)': stats['Benchmark_Return'],
            'Alpha (%)': stats['Alpha'],
            'Benchmark Sharpe': stats['Benchmark_Sharpe']
        }
    
    df = pd.DataFrame(summary).T
    
    print("\n" + "="*120)
    print("COMPREHENSIVE PERFORMANCE SUMMARY")
    print("="*120)
    print(df.to_string())
    print("="*120)

    df.to_csv('performance_summary.csv')
    print("\nSaved to: performance_summary.csv")
    
    return df

def plot_metric_comparison(results_dict, metrics=['CAGR', 'Sharpe', 'MaxDD', 'Alpha']):
    """Compare key metrics across periods"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    periods = list(results_dict.keys())
    
    metric_map = {
        'CAGR': 'CAGR',
        'Sharpe': 'Sharpe',
        'MaxDD': 'MaxDD',
        'Alpha': 'Alpha'
    }
    
    for idx, metric in enumerate(metrics):
        if idx >= 4:
            break
        
        strategy_vals = []
        benchmark_vals = []
        
        for name in periods:
            res = results_dict[name]
            if res is None:
                continue
            
            if metric == 'CAGR':
                strategy_vals.append(res['stats']['CAGR'])
                benchmark_vals.append(res['stats']['Benchmark_CAGR'])
            elif metric == 'Sharpe':
                strategy_vals.append(res['stats']['Sharpe'])
                benchmark_vals.append(res['stats']['Benchmark_Sharpe'])
            elif metric == 'MaxDD':
                strategy_vals.append(res['stats']['MaxDD'])
                benchmark_vals.append(res['stats']['Benchmark_MaxDD'])
            elif metric == 'Alpha':
                strategy_vals.append(res['stats']['Alpha'])
                benchmark_vals.append(0)  # Benchmark alpha is always 0
        
        x = np.arange(len(periods))
        width = 0.35
        
        ax = axes[idx]
        ax.bar(x - width/2, strategy_vals, width, label='Strategy', color='steelblue')
        ax.bar(x + width/2, benchmark_vals, width, label='NIFTY 50', color='coral')
        
        ax.set_ylabel(metric)
        ax.set_title(f'{metric} Comparison', fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(periods, rotation=45, ha='right')
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for i, v in enumerate(strategy_vals):
            ax.text(i - width/2, v, f'{v:.1f}', ha='center', va='bottom', fontsize=8)
        for i, v in enumerate(benchmark_vals):
            ax.text(i + width/2, v, f'{v:.1f}', ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    plt.savefig('metric_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

In [43]:


def evaluate_model_predictions(panel, sym_arrays, periods):
    """Evaluate model prediction accuracy across different periods"""
    results = {}
    
    for period_name, (start, end) in periods.items():
        period_data = panel[(panel['date'] >= start) & (panel['date'] <= end)]
        
        if len(period_data) == 0:
            continue
        

        valid = period_data.dropna(subset=['y', 'pred_xgb', 'pred_lstm', 'score'])
        
        if len(valid) == 0:
            continue

        xgb_metrics = calculate_metrics(valid['y'], valid['pred_xgb'], 'XGB_')
        lstm_metrics = calculate_metrics(valid['y'], valid['pred_lstm'], 'LSTM_')
        ensemble_metrics = calculate_metrics(valid['y'], valid['score'], 'Ensemble_')
        
        results[period_name] = {**xgb_metrics, **lstm_metrics, **ensemble_metrics}
    
    df = pd.DataFrame(results).T
    
    print("\n" + "="*100)
    print("MODEL PREDICTION PERFORMANCE")
    print("="*100)
    print(df.to_string())
    print("="*100)

    df.to_csv('model_performance.csv')
    
    return df

def plot_prediction_scatter(panel, period_name, start_date, end_date):
    """Plot actual vs predicted returns"""
    period_data = panel[(panel['date'] >= start_date) & (panel['date'] <= end_date)]
    valid = period_data.dropna(subset=['y', 'score']).sample(min(5000, len(period_data)))
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    models = [('pred_xgb', 'XGBoost'), ('pred_lstm', 'LSTM'), ('score', 'Ensemble')]
    
    for idx, (pred_col, model_name) in enumerate(models):
        ax = axes[idx]
        
        ax.scatter(valid['y'], valid[pred_col], alpha=0.3, s=10)
        ax.plot([-0.2, 0.2], [-0.2, 0.2], 'r--', linewidth=2, label='Perfect Prediction')
    
        r2 = r2_score(valid['y'], valid[pred_col])
        
        ax.set_xlabel('Actual Return')
        ax.set_ylabel('Predicted Return')
        ax.set_title(f'{model_name} - {period_name}\nR² = {r2:.4f}', fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'prediction_scatter_{period_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

def analyze_feature_importance(xgb_model):
    """Analyze and plot feature importance"""
    importance = pd.DataFrame({
        'Feature': FEATURES,
        'Importance': xgb_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\n" + "="*60)
    print("FEATURE IMPORTANCE (XGBoost)")
    print("="*60)
    print(importance.to_string(index=False))
    print("="*60)

    plt.figure(figsize=(10, 8))
    plt.barh(importance['Feature'], importance['Importance'])
    plt.xlabel('Importance')
    plt.title('Feature Importance (XGBoost)', fontweight='bold', fontsize=14)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return importance


In [44]:


def main_pipeline():
    """Main execution pipeline with profit optimizations"""
    
    print("\n" + "="*100)
    print(" ENHANCED STOCK TRADING SYSTEM - FULL 2025 PAPER TRADING")
    print("="*100)

    print("\n[1/10]  Loading index data...")
    idx_df = yf.download(CFG["index"], start=CFG["start"], end=CFG["end"], 
                        progress=False, auto_adjust=False)
    idx_df = normalize_yf_columns(idx_df)
    idx_close = idx_df["Close"].dropna()
    idx_close.index = pd.to_datetime(idx_close.index)
    print(f"   Index data: {len(idx_close)} days")
    
  
    print("\n[2/10]  Downloading stock data (2025 updated list)...")
    data = download_ohlcv(SYMBOLS, CFG["min_days"])
    
    if len(data) < 30:
        raise RuntimeError("Too few symbols. Check data quality.")
    print(f"   Successfully loaded {len(data)} stocks")
 
    print("\n[3/10] Building enhanced features...")
    sym_arrays, panel = build_symbol_arrays(data)
    print(f"   Symbols: {len(sym_arrays)} | Panel rows: {len(panel):,}")
    print(f"   Features: {len(FEATURES)}")
    

    print("\n[4/10] Training XGBoost...")
    train_end = pd.to_datetime(CFG["train_end"])
    tr = panel[panel["date"] <= train_end].dropna(subset=FEATURES + ["y"])
    print(f"   Training samples: {len(tr):,}")
    
    xgb = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.015,
        max_depth=7,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.5,
        reg_alpha=0.5,
        objective="reg:squarederror",
        random_state=CFG["seed"],
        n_jobs=-1,
        verbosity=0
    )
    
    xgb.fit(tr[FEATURES].values, tr["y"].values)
    panel["pred_xgb"] = xgb.predict(panel[FEATURES].values)
    print("   XGBoost training complete")
  
    print("\n[5/10] Training LSTM...")
    trX = panel.loc[panel["date"] <= train_end, FEATURES].values.astype(np.float32)
    mu = trX.mean(axis=0, keepdims=True)
    sig = trX.std(axis=0, keepdims=True) + 1e-6
    
    pairs = make_pairs(sym_arrays)
    
    train_ds = SeqSliceDataset(sym_arrays, pairs, start_date="2010-01-01", 
                              end_date=CFG["train_end"], standardize=(mu, sig))
    val_ds = SeqSliceDataset(sym_arrays, pairs, start_date=CFG["val_start"],
                            end_date=CFG["val_end"], standardize=(mu, sig))
    
    train_loader = DataLoader(train_ds, batch_size=CFG["lstm_batch"], 
                             shuffle=True, num_workers=0, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=CFG["lstm_batch"], 
                           shuffle=False, num_workers=0)
    
    print(f"   Training samples: {len(train_ds):,} | Validation: {len(val_ds):,}")
    model = train_lstm(train_loader, val_loader, n_features=len(FEATURES))
    print("   LSTM training complete")
    

    print("\n[6/10] Generating LSTM predictions...")
    pred_ds = SeqSliceDataset(sym_arrays, pairs, start_date="2010-01-01",
                             end_date=CFG["paper_end"], standardize=(mu, sig))
    pred_loader = DataLoader(pred_ds, batch_size=1024, shuffle=False, num_workers=0)
    pred_lstm_df = predict_lstm(model, pred_loader)
    print("    Predictions generated")
    

    print("\n[7/10] Creating optimized ensemble...")
    panel2 = panel.merge(pred_lstm_df, on=["date", "symbol"], how="left")
    panel2["pred_lstm"] = panel2["pred_lstm"].fillna(panel2["pred_xgb"])
    panel2["score"] = CFG["w_xgb"] * panel2["pred_xgb"] + CFG["w_lstm"] * panel2["pred_lstm"]
    print("   Ensemble created")
 
    print("\n[8/10] Running backtests...")
    
    results = {}
    
    results['Validation'] = run_backtest_enhanced(
        panel2, sym_arrays, idx_close,
        CFG["val_start"], CFG["val_end"], "Validation"
    )
    
    results['Testing'] = run_backtest_enhanced(
        panel2, sym_arrays, idx_close,
        CFG["test_start"], CFG["test_end"], "Testing"
    )
    
    results['Paper_Trading_2025'] = run_backtest_enhanced(
        panel2, sym_arrays, idx_close,
        CFG["paper_start"], CFG["paper_end"], "Paper_Trading_2025"
    )
    

    print("\n[9/10] Analyzing results...")
    perf_summary = create_performance_summary(results)
    trade_analysis = analyze_trades(results)
    analyze_holdings(results)
    
    periods = {
        'Training': (CFG["start"], CFG["train_end"]),
        'Validation': (CFG["val_start"], CFG["val_end"]),
        'Testing': (CFG["test_start"], CFG["test_end"]),
        'Paper_2025': (CFG["paper_start"], CFG["paper_end"])
    }
    model_perf = evaluate_model_predictions(panel2, sym_arrays, periods)
    feat_imp = analyze_feature_importance(xgb)
    

    print("\n[10/10] Generating visualizations...")
    plot_equity_curves(results)
    plot_monthly_returns(results)
    plot_drawdown(results)
    plot_returns_distribution(results)
    plot_rolling_metrics(results)
    plot_metric_comparison(results)
    
    for period_name, (start, end) in periods.items():
        if period_name != 'Training':
            plot_prediction_scatter(panel2, period_name, start, end)
    
    print("\n" + "="*100)
    print(" PIPELINE COMPLETE!")
    print("="*100)
    
 
    print("\n PERFORMANCE SUMMARY:")
    print("-" * 100)
    for period_name, res in results.items():
        if res and res.get('stats'):
            stats = res['stats']
            print(f"\n{period_name}:")
            print(f"  Return:  {stats['Total_Return']:>8.2f}% | Sharpe: {stats['Sharpe']:>6.2f} | "
                  f"MaxDD: {stats['MaxDD']:>7.2f}% | Alpha: {stats['Alpha']:>7.2f}%")
            print(f"  Trades:  {stats['Num_Trades']:>8d} | Cost: {stats['Total_Cost']:>6.2f}% | "
                  f"WinRate: {stats['Win_Rate']:>5.1f}%")

    if results.get('Paper_Trading_2025'):
        stats_2025 = results['Paper_Trading_2025']['stats']
        print("\n" + "="*100)
        print("2025 FULL YEAR PAPER TRADING RESULTS")
        print("="*100)
        print(f"\nPeriod: {stats_2025['Start']} to {stats_2025['End']}")
        print(f"Total Return: {stats_2025['Total_Return']:.2f}%")
        print(f"CAGR: {stats_2025['CAGR']:.2f}%")
        print(f"Sharpe Ratio: {stats_2025['Sharpe']:.2f}")
        print(f"Max Drawdown: {stats_2025['MaxDD']:.2f}%")
        print(f"Alpha vs NIFTY: {stats_2025['Alpha']:.2f}%")
        
     
        initial = 1000000
        final_value = initial * stats_2025['Final_Portfolio_Value']
        profit = final_value - initial
        
        print(f"\n On ₹10,00,000 investment:")
        print(f"   Final Value: ₹{final_value:,.0f}")
        print(f"   Profit/Loss: ₹{profit:,.0f}")
    
    return {
        'results': results,
        'panel': panel2,
        'sym_arrays': sym_arrays,
        'xgb_model': xgb,
        'lstm_model': model,
        'performance_summary': perf_summary,
        'model_performance': model_perf,
        'feature_importance': feat_imp
    }

In [48]:

def export_2025_transactions(results):
    """Export detailed 2025 transactions"""
    all_transactions = []
    
    for period_name, res in results.items():
        if res is None or not res.get('detailed_transactions'):
            continue
        
        for trans in res['detailed_transactions']:
            trans_copy = trans.copy()
            trans_copy['period'] = period_name
            trans_copy['sector'] = get_sector(trans['symbol'])
            all_transactions.append(trans_copy)
    
    if all_transactions:
        df = pd.DataFrame(all_transactions)
        df = df.sort_values(['period', 'date', 'action'])
        df.to_csv('detailed_transactions_2025.csv', index=False)
        print(f"\n Exported {len(all_transactions)} transactions to: detailed_transactions_2025.csv")
        
  
        paper_trades = df[df['period'] == 'Paper_Trading_2025']
        if len(paper_trades) > 0:
            print("\n" + "="*100)
            print(" 2025 FULL YEAR TRANSACTIONS")
            print("="*100)
            
            buys = paper_trades[paper_trades['action'] == 'BUY']
            sells = paper_trades[paper_trades['action'] == 'SELL']
            stops = paper_trades[paper_trades['action'] == 'STOP_LOSS']
            
            print(f"\nTransaction Summary:")
            print(f"   Total Buys:       {len(buys):3d}")
            print(f"   Total Sells:      {len(sells):3d}")
            print(f"   Stop-Loss Hits:   {len(stops):3d}")
            
            if len(buys) > 0:
                print(f"\nLatest Buys:")
                for _, row in buys.tail(10).iterrows():
                    print(f"   {row['date'].strftime('%Y-%m-%d')} | {row['symbol']:15s} | "
                          f"Weight: {row['weight']*100:5.2f}% | {row['sector']}")
            
            if len(stops) > 0:
                print(f"\nStop-Loss Triggers:")
                for _, row in stops.iterrows():
                    print(f"   {row['date'].strftime('%Y-%m-%d')} | {row['symbol']:15s} | "
                          f"Loss: {row.get('return', 0)*100:5.2f}%")
        
        return df
    
    return None


def analyze_2025_full_year(results):
    """Comprehensive 2025 analysis"""
    paper_res = results.get('Paper_Trading_2025')
    
    if not paper_res:
        print("\nNo 2025 results available")
        return
    
    stats = paper_res['stats']
    
    print("\n" + "="*100)
    print("2025 FULL YEAR ANALYSIS")
    print("="*100)
    
    print(f"\nPerformance:")
    print(f"   Total Return:     {stats['Total_Return']:>10.2f}%")
    print(f"   CAGR:             {stats['CAGR']:>10.2f}%")
    print(f"   Sharpe Ratio:     {stats['Sharpe']:>10.2f}")
    print(f"   Max Drawdown:     {stats['MaxDD']:>10.2f}%")
    print(f"   Win Rate:         {stats['Win_Rate']:>10.2f}%")
    
    print(f"\n Trading:")
    print(f"   Trades:           {stats['Num_Trades']:>10d}")
    print(f"   Total Costs:      {stats['Total_Cost']:>10.2f}%")
    print(f"   Alpha:            {stats['Alpha']:>10.2f}%")
    

    if paper_res.get('holdings_history'):
        latest = paper_res['holdings_history'][-1]
        print(f"\n Current Portfolio ({latest['date'].strftime('%Y-%m-%d')}):")
        for i, stock in enumerate(latest['holdings'], 1):
            print(f"   {i:2d}. {stock:15s} | {get_sector(stock)}")
        
  
        sectors = {}
        for stock in latest['holdings']:
            sector = get_sector(stock)
            sectors[sector] = sectors.get(sector, 0) + 1
        
        print(f"\n   Sectors:")
        for sector, count in sorted(sectors.items(), key=lambda x: -x[1]):
            pct = (count / len(latest['holdings'])) * 100
            print(f"   {sector:15s} : {count:2d} ({pct:5.1f}%)")
    

    print(f"\n📅 Monthly Performance:")
    equity_series = pd.Series(paper_res["equity"], index=pd.DatetimeIndex(paper_res["dates"]))
    monthly = equity_series.resample('ME').last().pct_change().dropna() * 100
    
    if len(monthly) > 0:
        for date, ret in monthly.items():

            print(f"   {date.strftime('%b %Y'):11s} | {ret:>7.2f}%")
        
        print(f"\n   Best:  {monthly.max():>7.2f}%")
        print(f"   Worst: {monthly.min():>7.2f}%")
        print(f"   Avg:   {monthly.mean():>7.2f}%")
        print(f"   Positive: {(monthly > 0).sum()}/{len(monthly)} months")


def final_recommendations(results):
    """Generate actionable recommendations"""
    paper_res = results.get('Paper_Trading_2025')
    
    if not paper_res:
        return
    
    stats = paper_res['stats']
    
    print("\n" + "="*100)
    print("RECOMMENDATIONS")
    print("="*100)
    

    if stats['Total_Return'] > 15:
        print(f"\nEXCELLENT: {stats['Total_Return']:.2f}% return!")
    elif stats['Total_Return'] > 0:
        print(f"\nGOOD: {stats['Total_Return']:.2f}% positive")
    else:
        print(f"\nNEEDS WORK: {stats['Total_Return']:.2f}% loss")
    

    issues = []
    
    if stats['Total_Cost'] > 2.0:
        issues.append(f"High costs ({stats['Total_Cost']:.2f}%) - Reduce rebalancing")
    
    if abs(stats['MaxDD']) > 25:
        issues.append(f"High drawdown ({stats['MaxDD']:.1f}%) - Tighten stop-loss")
    
    if stats['Sharpe'] < 1.0:
        issues.append(f"Low Sharpe ({stats['Sharpe']:.2f}) - Add volatility filters")
    
    if stats['Alpha'] < 0:
        issues.append(f"Negative alpha ({stats['Alpha']:.2f}%) - Improve stock selection")
    
    if issues:
        print("\nAction Items:")
        for i, issue in enumerate(issues, 1):
            print(f"   {i}. {issue}")
    else:
        print("\nNo major issues! Fine-tune and continue.")

    print("\nPotential Improvements:")
    current = stats['Total_Return']
    cost_save = min(stats['Total_Cost'] * 0.5, 2.0)
    signal_improve = abs(current) * 0.15
    risk_improve = abs(stats['MaxDD']) * 0.20
    
    projected = current+cost_save+signal_improve+risk_improve
    
    print(f"   Current:      {current:>8.2f}%")
    print(f"   + Cost save:  {cost_save:>8.2f}%")
    print(f"   + Better AI:  {signal_improve:>8.2f}%")
    print(f"   + Risk mgmt:  {risk_improve:>8.2f}%")
    print(f"   " + "-" * 30)
    print(f"   Projected:    {projected:>8.2f}%")
    

    for capital in [100000, 1000000]:
        current_profit = capital * (current / 100)
        projected_profit = capital * (projected / 100)
        print(f"\n   ₹{capital:,}:")
        print(f"      Current:   ₹{current_profit:>12,.0f}")
        print(f"      Potential: ₹{projected_profit:>12,.0f}")


if __name__ == "__main__":
    print("\n" + "="*100)
    print("2025 DETAILED ANALYSIS")
    print("="*100)
 
    trans_df = export_2025_transactions(pipeline_output['results'])
  
    analyze_2025_full_year(pipeline_output['results'])
 
    final_recommendations(pipeline_output['results'])
    
    print("\n" + "="*100)
    print("2025 ANALYSIS COMPLETE!")
    print("="*100)
    
    print("\nFiles Generated:")
    print("   detailed_transactions_2025.csv")
    print("   All visualization PNGs")
    print("   Performance CSVs")
    
    print("\n DISCLAIMER:")
    print("   Past performance ≠ future results")
    print("   Start with small capital")
    print("   Consult professionals before live trading")
    
    print("\n" + "="*100)
    print("="*100)


2025 DETAILED ANALYSIS

 Exported 1177 transactions to: detailed_transactions_2025.csv

 2025 FULL YEAR TRANSACTIONS

Transaction Summary:
   Total Buys:       295
   Total Sells:      279
   Stop-Loss Hits:     1

Latest Buys:
   2025-12-05 | BHARTIARTL.NS   | Weight:  6.47% | Telecom
   2025-12-05 | AXISBANK.NS     | Weight:  6.61% | Banking
   2025-12-05 | ICICIBANK.NS    | Weight:  6.38% | Banking
   2025-12-12 | SHRIRAMFIN.NS   | Weight:  4.74% | NBFC
   2025-12-12 | HEROMOTOCO.NS   | Weight:  3.97% | Auto
   2025-12-12 | HCLTECH.NS      | Weight:  4.96% | IT
   2025-12-12 | TECHM.NS        | Weight:  6.03% | IT
   2025-12-12 | HINDALCO.NS     | Weight:  5.98% | Metals
   2025-12-12 | EICHERMOT.NS    | Weight:  5.36% | Auto
   2025-12-12 | INFY.NS         | Weight:  5.84% | IT

Stop-Loss Triggers:
   2025-03-11 | INDUSINDBK.NS   | Loss: -27.16%

2025 FULL YEAR ANALYSIS

Performance:
   Total Return:           2.26%
   CAGR:                   2.35%
   Sharpe Ratio:           0.63
